# Spotify Music Recommendation System
## Notebook 06 — Evaluation

**Purpose:** Measure recommendation quality using three proxy metrics since we have no direct user listening history.

## Table of Contents
1. [Setup & Imports](#1-setup--imports)
2. [Load Data](#2-load-data)
3. [Evaluation Strategy](#3-evaluation-strategy)
4. [Metric 1 — Genre Precision@K](#4-metric-1--genre-precisionk)
5. [Metric 2 — Popularity Coherence](#5-metric-2--popularity-coherence)
6. [Metric 3 — Similarity Score Distribution](#6-metric-3--similarity-score-distribution)
7. [Case Studies](#7-case-studies)
8. [Final Evaluation Report](#8-final-evaluation-report)

## 1. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import ast, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

DATA_DIR  = Path('../data/processed')
RAW_DIR   = Path('../data/raw')
MODEL_DIR = Path('../models')

## 2. Load Data

In [ ]:
with open(MODEL_DIR / 'recommender_payload.pkl', 'rb') as f:
    payload = pickle.load(f)

feature_matrix  = payload['feature_matrix']
track_index     = payload['track_index']
feature_columns = payload['feature_columns']

data_genre  = pd.read_csv(RAW_DIR  / 'data_w_genres.csv')
song_artist = pd.read_csv(DATA_DIR / 'song_and_artists_clean.csv')

print(f'Feature matrix : {feature_matrix.shape}')
print(f'Track index    : {track_index.shape}')
print(f'Eval songs     : {song_artist.shape}')

In [ ]:
# Recreate recommendation function
def get_recommendations(song_name, top_n=10):
    lower   = song_name.lower()
    exact   = track_index[track_index['name'].str.lower() == lower]
    partial = track_index[track_index['name'].str.lower().str.contains(lower, na=False, regex=False)]
    matches = exact if not exact.empty else partial
    if matches.empty:
        return None, None
    q = matches.loc[matches['popularity'].idxmax()]
    q_idx = q.name
    sims  = cosine_similarity(feature_matrix[q_idx].reshape(1,-1), feature_matrix).flatten()
    result = track_index.copy()
    result['similarity'] = sims
    result = result[result.index != q_idx]
    top = result.nlargest(top_n, 'similarity').reset_index(drop=True)
    top.index += 1
    return top[['name', 'artists', 'year', 'popularity', 'similarity']], q

# Build artist -> genres lookup (handle [] empty genres)
def parse_genres(s):
    try:
        r = ast.literal_eval(str(s))
        return set(r) if isinstance(r, list) else set()
    except Exception:
        return set()

genre_lookup = data_genre.set_index('artists')['genres'].apply(parse_genres).to_dict()

def get_genres(artists_str):
    genres = set()
    for a in str(artists_str).split(', '):
        genres |= genre_lookup.get(a.strip(), set())
    return genres

track_index['genres'] = track_index['artists'].apply(get_genres)
has_genre = track_index['genres'].apply(len) > 0
print(f'Tracks with genre info: {has_genre.sum():,} / {len(track_index):,}')

## 3. Evaluation Strategy

We have no explicit user-listening history, so we use **proxy metrics**.

| Metric | Assumption | Measures |
|---|---|---|
| **Genre Precision@K** | Same-genre songs are relevant | Genre coherence of top-K recommendations |
| **Popularity Coherence** | Users want songs of similar popularity | Popularity gap between seed and recommendations |
| **Similarity Distribution** | Higher cosine = better audio match | How tight the top-10 similarity scores are |

We sample 100 seed songs (with genre info, popularity 20–80) and run each through the recommender.

In [ ]:
# Select 100 representative seed songs
seeds = track_index[
    track_index['genres'].apply(len) > 0
    & track_index['popularity'].between(20, 80)
].sample(100, random_state=42)

print(f'Evaluation seed songs: {len(seeds)}')
seeds[['name', 'artists', 'year', 'popularity']].head(5)

## 4. Metric 1 — Genre Precision@K

For each seed song, we check: **what fraction of the top-K recommendations share a genre with the seed?**

Example: if seed is a 'pop' song and 7 of 10 recommendations also have 'pop' in their genres, P@10 = 0.70

In [ ]:
def precision_at_k(seed_genres, rec_artists_list, k=10):
    if not seed_genres:
        return None
    hits = sum(1 for a in rec_artists_list[:k] if get_genres(a) & seed_genres)
    return hits / k

# Evaluate across all seeds
p10_scores = []
for _, row in seeds.iterrows():
    recs, _ = get_recommendations(row['name'], top_n=10)
    if recs is None:
        continue
    p = precision_at_k(row['genres'], recs['artists'].tolist(), k=10)
    if p is not None:
        p10_scores.append(p)

print(f'Seeds evaluated     : {len(p10_scores)}')
print(f'Mean Precision@10   : {np.mean(p10_scores):.3f}')
print(f'Median Precision@10 : {np.median(p10_scores):.3f}')

In [ ]:
# Precision at different K values
k_values = [1, 3, 5, 10, 20]
mean_pk  = []
for k in k_values:
    scores = []
    for _, row in seeds.iterrows():
        recs, _ = get_recommendations(row['name'], top_n=k)
        if recs is None: continue
        p = precision_at_k(row['genres'], recs['artists'].tolist(), k=k)
        if p is not None: scores.append(p)
    mean_pk.append(np.mean(scores) if scores else 0)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(k_values, mean_pk, marker='o', color='steelblue', linewidth=2)
ax.set_title('Mean Precision@K (Genre Overlap)', fontweight='bold')
ax.set_xlabel('K (number of recommendations)')
ax.set_ylabel('Precision@K')
ax.set_ylim(0, 1)
for k, p in zip(k_values, mean_pk):
    ax.annotate(f'{p:.2f}', (k, p), textcoords='offset points', xytext=(0, 8),
                ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## 5. Metric 2 — Popularity Coherence

How close is the popularity of recommended songs to the seed song? A good recommender should suggest songs at a similar popularity tier.

In [ ]:
pop_deltas = []
for _, row in seeds.iterrows():
    recs, _ = get_recommendations(row['name'], top_n=10)
    if recs is None: continue
    pop_deltas.extend((recs['popularity'] - row['popularity']).abs().tolist())

pop_deltas = np.array(pop_deltas)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(pop_deltas, bins=30, color='mediumpurple', edgecolor='none')
ax.axvline(pop_deltas.mean(), color='red', linewidth=1.5, linestyle='--',
           label=f'Mean delta = {pop_deltas.mean():.1f}')
ax.set_title('Popularity Gap: Seed vs Recommendations', fontweight='bold')
ax.set_xlabel('|Popularity(recommendation) - Popularity(seed)|')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Mean popularity gap    : {pop_deltas.mean():.2f}')
print(f'Within ±10 popularity  : {(pop_deltas <= 10).mean()*100:.1f}%')
print(f'Within ±20 popularity  : {(pop_deltas <= 20).mean()*100:.1f}%')

## 6. Metric 3 — Similarity Score Distribution

The cosine similarity of the top-10 recommendations across all seed songs. Higher values mean the recommender finds tracks that are genuinely close in audio feature space.

In [ ]:
top10_sims = []
for _, row in seeds.iterrows():
    recs, _ = get_recommendations(row['name'], top_n=10)
    if recs is not None:
        top10_sims.extend(recs['similarity'].tolist())

top10_sims = np.array(top10_sims)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(top10_sims, bins=40, color='seagreen', edgecolor='none')
ax.axvline(top10_sims.mean(), color='red', linewidth=1.5, linestyle='--',
           label=f'Mean = {top10_sims.mean():.3f}')
ax.set_title('Cosine Similarity of Top-10 Recommendations (all seeds)', fontweight='bold')
ax.set_xlabel('Cosine Similarity')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Mean similarity  : {top10_sims.mean():.4f}')
print(f'Min similarity   : {top10_sims.min():.4f}')
print(f'% sims > 0.90    : {(top10_sims > 0.90).mean()*100:.1f}%')

## 7. Case Studies

In [ ]:
case_songs = ['White Christmas', 'Lose Yourself', 'Nothing Else Matters']

for song in case_songs:
    recs, q = get_recommendations(song, top_n=8)
    if recs is None:
        print(f'{song}: not found\n'); continue
    print(f'\n{"-"*60}')
    seed_genres = get_genres(q['artists'])
    print(f'SEED : {q["name"]} | {q["artists"]} | {int(q["year"])} | Pop={int(q["popularity"])}')
    print(f'Seed genres: {seed_genres if seed_genres else "none found"}')
    print(f'{"-"*60}')
    print(recs.to_string())

## 8. Final Evaluation Report

In [ ]:
print('='*55)
print('EVALUATION REPORT')
print('='*55)
print(f'Dataset          : {len(track_index):,} tracks')
print(f'Feature vector   : {feature_matrix.shape[1]} dimensions')
print(f'Seed songs tested: {len(seeds)}')
print()
print('Metric 1 — Genre Precision@K')
for k, p in zip(k_values, mean_pk):
    bar = '#' * int(p * 20)
    print(f'  P@{k:<3} = {p:.3f}  [{bar:<20}]')
print()
print('Metric 2 — Popularity Coherence')
print(f'  Mean |popularity gap| = {pop_deltas.mean():.2f}')
print(f'  Within ±10           = {(pop_deltas <= 10).mean()*100:.1f}%')
print()
print('Metric 3 — Cosine Similarity')
print(f'  Mean top-10 sim      = {top10_sims.mean():.4f}')
print(f'  % of recs > 0.90 sim = {(top10_sims > 0.90).mean()*100:.1f}%')
print('='*55)

### Interpretation

| Metric | Result | What it means |
|---|---|---|
| Genre Precision@10 | see above | What fraction of top-10 recs share a genre with the seed |
| Popularity gap | see above | How close recs are in popularity to the seed |
| Cosine similarity | see above | How tight the audio match is (higher = better) |

### What Could Be Improved
| Issue | Fix |
|---|---|
| 34% of tracks have no genre → genre metric covers less data | Add genre data via external API |
| Popularity bias in feature vector | Remove or lower-weight `popularity_norm` |
| Content-only (no user history) | Add collaborative filtering as a second pass |